# Description
Run `database_prep.ipynb` before running this script.

# Table of Contents
1. [Description](#description)
2. [Imports](#imports)
3. [Data Manipulations](#data-manipulations)
4. [Stats](#stats)  
4.1 [Basic Stats](#basic-stats)  
4.2 [Descriptive Stats](#desriptive-stats)  
5. [Main Analysis](#main-anlaysis)  
5.1 [ANCOVA](#ancova)


# Imports

In [1]:
import pandas as pd
import os
import numpy as np
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import fdrcorrection

# Basic statistical tests for significant features
from scipy import stats
import statsmodels.api as sm
from scipy.stats import shapiro, levene
import matplotlib.pyplot as plt
import seaborn as sns

# ANOVA
from patsy import dmatrix

In [7]:
root_dir = "/Users/emmanuelle.coutu-nadeau/Library/Mobile Documents/com~apple~CloudDocs/UdeM/MSc Psycho/LABO NED - Personal Drive/Code/GENiAL/"
og_data = os.path.join(root_dir, 'Data/Final/GENIAL-DB-preprocessed-RS.csv') # original data
og_data = pd.read_csv(og_data)

# Data Manipulations

Data Types

In [8]:
# Non numeric columns
non_numeric_cols = ['ParticipantID', 'EEG_attempted', 'EEG_site', 'Birthdate', 'EEG_date', 'Sex_at_birth', 'diag_unknown_specify', 'diag_other_specify', 'medication_at_EEG', 'RS_Rio_done', 'RS_Rio_code', 'RS_done', 'RS_code', 'TO_done', 'TO_code', 'GO_done', 'GO_code', 'VEP_done', 'VEP_code', 'AEP_done', 'AEP_code', 'AEP_randomization_file', 'NSP_done', 'NSP_code', 'VS_done', 'VS_code', 'MMN_done', 'MMN_code', 'Genetic_test_result', 'Genetic_status', 'Genome_version', 'Single_gene_testing', 'Fragile_X', 'Exome_panel_testing', 'family_member_type']

# Convert non-numeric columns to string type
for col in non_numeric_cols:
    og_data[col] = og_data[col].astype(str)


# Numeric columns
numeric_cols = [col for col in og_data.columns if col not in non_numeric_cols]

# Convert numeric columns to number, coercing errors to NaN
for col in numeric_cols:
    og_data[col] = pd.to_numeric(og_data[col], errors='coerce')


# Get all non-EEG columns
non_eeg_cols = ['ParticipantID', 'EEG_attempted', 'EEG_site', 'Birthdate', 'EEG_date', 'EEG_age', 'Sex_at_birth', 'diag_unknown_specify', 'diag_other_specify', 'medication_at_EEG', 'RS_Rio_done', 'RS_Rio_code', 'RS_done', 'RS_code', 'TO_done', 'TO_code', 'GO_done', 'GO_code', 'VEP_done', 'VEP_code', 'AEP_done', 'AEP_code', 'AEP_randomization_file', 'NSP_done', 'NSP_code', 'VS_done', 'VS_code', 'MMN_done', 'MMN_code', 'Genetic_test_result', 'Genetic_status', 'Affected_chromosome', 'Proximal_boundary', 'Distal_boundary', 'Genome_version', 'Single_gene_testing', 'Fragile_X', 'Exome_panel_testing', 'diag_control', 'diag_neurodev', 'diag_genetic_carrier', 'diag_unknown', 'diag_other', 'inheritance_denovo', 'inheritance_mothers_inherited', 'inheritance_fathers_inherited', 'inheritance_unknown', 'inheritance_mosaic', 'family_member_type', 'NVIQ_CIupr', 'ORASD_upr', 'SRS_CIupr', 'PdN_CIupr', 'sum_LOEUF_complete', 'diag_asd', 'diag_intel', 'diag_adhd', 'diag_fas', 'diag_learn', 'diag_comm', 'diag_motor', 'wais_date', 'wais_age', 'waisgrade2_norm___1', 'waisgrade2_norm___2', 'wais_bd_rgss', 'wais_sim_rgss', 'wais_matrix_rgss', 'wais_vocab_rgss', 'wais_vispuzz_rgss', 'wais_info_rgss', 'wais_globalapt_comp', 'wisc_date', 'wisc_norm_used', 'wisc_bd_ss', 'wisc_si_ss', 'wisc_mr_ss', 'wisc_vc_ss', 'wisc_vp_ss', 'wisc_in_ss', 'wisc_gai_is', 'ID']

# Get all EEG columns
eeg_cols = [col for col in og_data.columns if col.startswith('EEG_') and col not in non_eeg_cols]

# Make sure EEG features are numeric
for col in eeg_cols:
    og_data[col] = pd.to_numeric(og_data[col], errors='coerce')


Remove over 80% EEG features issing rows

In [9]:
# Calculate the percentage of missing values for each row
missing_percentage = og_data[eeg_cols].isnull().mean(axis=1)

# Keep only rows where less than 80% of EEG features are missing
no_missing_data = og_data[missing_percentage < 0.8].reset_index(drop=True)

print(f"Rows remaining after dropping those with >80% missing EEG data: {len(no_missing_data)}")


Rows remaining after dropping those with >80% missing EEG data: 80


In [10]:
diagnostic_groups_data = no_missing_data.copy()

# Add diagnostic group column
# 0: Control (diag_control = 1)
# 1: Neurodev only (diag_neurodev = 1 and diag_genetic_carrier = 0)
# 2: Genetic carrier (diag_genetic_carrier = 1)
diagnostic_groups_data['diagnostic_group'] = 0

# Set group 1: Neurodev only
diagnostic_groups_data.loc[(diagnostic_groups_data['diag_neurodev'] == 1) & (diagnostic_groups_data['diag_genetic_carrier'] == 0), 'diagnostic_group'] = 1

# Set group 2: Genetic carrier
diagnostic_groups_data.loc[diagnostic_groups_data['diag_genetic_carrier'] == 1, 'diagnostic_group'] = 2


In [11]:
# Drop EEG_Age and EEG_Sex columns
# We use Sex_at_birth and EEG_age
diagnostic_groups_data = diagnostic_groups_data.drop(['EEG_Age', 'EEG_Sex'], axis=1)


# Stats

In [12]:
# Group features by type
feature_groups = {
    'Offset': [f for f in diagnostic_groups_data.columns if 'offset' in f.lower()],
    'Exponent': [f for f in diagnostic_groups_data.columns if 'exponent' in f.lower()],
    'Delta': [f for f in diagnostic_groups_data.columns if 'delta' in f.lower()],
    'Theta': [f for f in diagnostic_groups_data.columns if 'theta' in f.lower()],
    'Alpha': [f for f in diagnostic_groups_data.columns if 'alpha' in f.lower()],
    'Beta': [f for f in diagnostic_groups_data.columns if 'beta' in f.lower()],
    'Low Gamma': [f for f in diagnostic_groups_data.columns if 'lowgamma' in f.lower()],
    'High Gamma': [f for f in diagnostic_groups_data.columns if 'highgamma' in f.lower()],
    'Relative': [f for f in diagnostic_groups_data.columns if 'relative' in f.lower()],
    'Periodic': [f for f in diagnostic_groups_data.columns if 'periodic' in f.lower()]
}

# Add "Other" category for features that don't fit existing categories
categorized_features = [f for group in feature_groups.values() for f in group]
other_features = [f for f in diagnostic_groups_data.columns if f not in categorized_features and f not in ['diagnostic_group']]
if other_features:
    feature_groups['Other'] = other_features

In [13]:
# Get all EEG columns
eeg_cols = [col for col in diagnostic_groups_data.columns if col.startswith('EEG_') and col not in non_eeg_cols]


### Basic Stats

In [14]:
print("\nBasic Statistical Tests for Significant Features")
print("-" * 50)

# Initialize lists to store features based on assumption violations
features_all_ok = []
features_normality_violated = []
features_homogeneity_violated = []

# Create files to store assumption results
with open(os.path.join(root_dir, 'Output/SPR-2025/assumptions_violated.txt'), 'w') as f_violated:
    f_violated.write("Features with violated assumptions:\n\n")
    
with open(os.path.join(root_dir, 'Output/SPR-2025/assumptions_ok.txt'), 'w') as f_ok:
    f_ok.write("Features with all assumptions met:\n\n")

for group_name, features in feature_groups.items():
    if not features:
        continue
        
    print(f"\n{group_name}:")
    print("-" * 30)
    
    for feature in eeg_cols:
        assumptions_violated = []
        
        # Create subplots for diagnostic plots
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
        fig.suptitle(f'Diagnostic Plots for {feature}')
        
        # ---------------------------#
        # 1. Normality tests per group
        # ---------------------------#
        print(f"\nFeature: {feature}")
        print("\nShapiro-Wilk Test Results (Normality):")
        normality_violated = False

        # Calculate overall data range for consistent binning
        all_data = diagnostic_groups_data[feature].dropna()
        data_min, data_max = all_data.min(), all_data.max()
        n_bins = min(50, int(np.sqrt(len(all_data))))  # Limit number of bins
    
        # Create a copy of the data for plotting
        plot_data = diagnostic_groups_data.copy()

        for group in diagnostic_groups_data['diagnostic_group'].unique():
            group_data = diagnostic_groups_data[diagnostic_groups_data['diagnostic_group'] == group][feature].dropna()
            stat, p_value = shapiro(group_data)
            if p_value < 0.05:
                normality_violated = True
            
            # Histogram per group
            try:
                sns.histplot(
                    data=group_data,
                    ax=ax1,
                    label=group,
                    alpha=0.3,
                    bins=n_bins,
                    binrange=(data_min, data_max),
                    stat='density'  # Use density instead of count
                )
            except ValueError as e:
                print(f"Warning: Could not create histogram for group {group}: {e}")
                continue
        
        if normality_violated:
            assumptions_violated.append("Normality")
            features_normality_violated.append(feature)
            print("⚠️ WARNING: Normality assumption violated (p < 0.05)")
            
        ax1.set_title('Histogram by Group')
        ax1.legend()
        
        # QQ Plot
        sm.graphics.qqplot(diagnostic_groups_data[feature].dropna(), line='45', ax=ax2)
        ax2.set_title('Q-Q Plot')
        
        # ---------------------------#
        # 2. Homogeneity of variance
        # ---------------------------#
        # Levene's test
        groups_data = [group_data for name, group_data in diagnostic_groups_data.groupby('diagnostic_group')[feature]]
        stat, p_value = levene(*groups_data)
        print(f"\nLevene's Test (Homogeneity of Variance):")
        
        if p_value < 0.05:
            assumptions_violated.append("Homogeneity of variance")
            features_homogeneity_violated.append(feature)
            print("⚠️ WARNING: Homogeneity of variance assumption violated (p < 0.05)")
        
        # Boxplot and Violin plot
        try:
            sns.boxplot(
                data=plot_data,
                x='diagnostic_group',
                y=feature,
                ax=ax3
            )
            ax3.set_title('Boxplot')
            
            sns.violinplot(
                data=plot_data,
                x='diagnostic_group',
                y=feature,
                ax=ax4
            )
            ax4.set_title('Violin Plot')
        except ValueError as e:
            print(f"Warning: Could not create box/violin plots for {feature}: {e}")
            ax3.text(0.5, 0.5, 'Plot creation failed', ha='center', va='center')
            ax4.text(0.5, 0.5, 'Plot creation failed', ha='center', va='center')

        plt.tight_layout()
        plt.savefig(os.path.join(root_dir, f'Output/SPR-2025/basic_stats/diagnostic_plots_{feature}.pdf'))
        plt.close()
        
        # Write results to appropriate file and update features_all_ok list
        if assumptions_violated:
            with open(os.path.join(root_dir, 'Output/SPR-2025/assumptions_violated.txt'), 'a') as f:
                f.write(f"{feature} ({group_name}):\n")
                for violation in assumptions_violated:
                    f.write(f"  - {violation}\n")
                f.write("\n")
        else:
            features_all_ok.append(feature)
            with open(os.path.join(root_dir, 'Output/SPR-2025/assumptions_ok.txt'), 'a') as f:
                f.write(f"{feature} ({group_name})\n")
        
        print("-" * 50)



Basic Statistical Tests for Significant Features
--------------------------------------------------

Offset:
------------------------------

Feature: EEG_Exponent-Frontal

Shapiro-Wilk Test Results (Normality):

Levene's Test (Homogeneity of Variance):
--------------------------------------------------

Feature: EEG_Exponent-Central

Shapiro-Wilk Test Results (Normality):

Levene's Test (Homogeneity of Variance):
⚠️ WARNING: Homogeneity of variance assumption violated (p < 0.05)
--------------------------------------------------

Feature: EEG_Exponent-temporal-r

Shapiro-Wilk Test Results (Normality):

Levene's Test (Homogeneity of Variance):
--------------------------------------------------

Feature: EEG_Exponent-temporal-l

Shapiro-Wilk Test Results (Normality):

Levene's Test (Homogeneity of Variance):
--------------------------------------------------

Feature: EEG_Exponent-parietal-r

Shapiro-Wilk Test Results (Normality):
⚠️ WARNING: Normality assumption violated (p < 0.05)

Le

### Adjust Normality & Homogeneity

In [15]:
still_non_normal = []
features_to_use = []  # Final list of usable features (transformed or original)

# Handle features with violated normality assumptions
print("Handling features with violated normality...")

# Apply log transformation to features that violated normality
for feature in features_normality_violated:
    print(f"\nApplying log transformation to {feature}")
    
    # Add small constant to handle zeros/negative values
    min_val = diagnostic_groups_data[feature].min()
    if min_val <= 0:
        offset = abs(min_val) + 1
        print(f"Added offset of {offset:.3f} to avoid log(0) for {feature}")
        diagnostic_groups_data[f'{feature}_log'] = np.log(diagnostic_groups_data[feature] + offset)
    else:
        diagnostic_groups_data[f'{feature}_log'] = np.log(diagnostic_groups_data[feature])
    
    # Test normality of transformed data
    feature_passes_normality = True
    for name, group in diagnostic_groups_data.groupby('diagnostic_group'):
        stat, p_value = shapiro(group[f'{feature}_log'].dropna())
        print(f"\nShapiro-Wilk test for {name} after log transformation:")
        print(f"statistic={stat:.3f}, p-value={p_value:.3f}")
        
        if p_value < 0.05:
            print(f"⚠️ Note: Log transformation did not achieve normality for {name}")
            still_non_normal.append((feature, name))
            feature_passes_normality = False

    if feature_passes_normality:
        features_to_use.append(f'{feature}_log')
    else:
        features_to_use.append(feature)  # Use original for now, but flag for non-parametric later

# For features that passed all assumptions originally
for feature in features_all_ok:
    if feature not in features_to_use:
        features_to_use.append(feature)

# Handle features with violated homogeneity
print("\nFeatures with violated homogeneity of variance:")
for feature in features_homogeneity_violated:
    print(f"- {feature}")
print("\nNote: Consider using Welch's ANOVA or non-parametric tests for these features")

# Summary
print("\n✅ Final list of features to use in models:")
for feat in features_to_use:
    print(f"- {feat}")


Handling features with violated normality...

Applying log transformation to EEG_Exponent-parietal-r

Shapiro-Wilk test for 0 after log transformation:
statistic=0.935, p-value=0.091

Shapiro-Wilk test for 1 after log transformation:
statistic=0.905, p-value=0.015
⚠️ Note: Log transformation did not achieve normality for 1

Shapiro-Wilk test for 2 after log transformation:
statistic=0.932, p-value=0.096

Applying log transformation to EEG_Exponent-parietal-l

Shapiro-Wilk test for 0 after log transformation:
statistic=0.721, p-value=0.000
⚠️ Note: Log transformation did not achieve normality for 0

Shapiro-Wilk test for 1 after log transformation:
statistic=0.935, p-value=0.083

Shapiro-Wilk test for 2 after log transformation:
statistic=0.906, p-value=0.025
⚠️ Note: Log transformation did not achieve normality for 2

Applying log transformation to EEG_Offset-temporal-r
Added offset of 1.201 to avoid log(0) for EEG_Offset-temporal-r

Shapiro-Wilk test for 0 after log transformation:
st

/var/folders/7j/mcx19g313_vgs3_tv_rpqmrw0000gn/T/ipykernel_54504/4121215052.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  diagnostic_groups_data[f'{feature}_log'] = np.log(diagnostic_groups_data[feature])
/var/folders/7j/mcx19g313_vgs3_tv_rpqmrw0000gn/T/ipykernel_54504/4121215052.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  diagnostic_groups_data[f'{feature}_log'] = np.log(diagnostic_groups_data[feature])
/var/folders/7j/mcx19g313_vgs3_tv_rpqmrw0000gn/T/ipykernel_54504/4121215052.py:18: PerformanceWarning: DataFra


Shapiro-Wilk test for 0 after log transformation:
statistic=0.954, p-value=0.263

Shapiro-Wilk test for 1 after log transformation:
statistic=0.954, p-value=0.251

Shapiro-Wilk test for 2 after log transformation:
statistic=0.954, p-value=0.309

Applying log transformation to EEG_relative-LowGamma-WholeBrain

Shapiro-Wilk test for 0 after log transformation:
statistic=0.961, p-value=0.397

Shapiro-Wilk test for 1 after log transformation:
statistic=0.943, p-value=0.131

Shapiro-Wilk test for 2 after log transformation:
statistic=0.947, p-value=0.210

Applying log transformation to EEG_relative-HighGamma-Frontal

Shapiro-Wilk test for 0 after log transformation:
statistic=0.990, p-value=0.994

Shapiro-Wilk test for 1 after log transformation:
statistic=0.949, p-value=0.188

Shapiro-Wilk test for 2 after log transformation:
statistic=0.977, p-value=0.821

Applying log transformation to EEG_relative-HighGamma-Central

Shapiro-Wilk test for 0 after log transformation:
statistic=0.936, p-v

### Desriptive Stats

Summarize data

In [16]:
# Get summary statistics for features we'll use in main analysis
features_summary = diagnostic_groups_data[features_to_use].agg(['min', 'max', 'mean']).round(2)

# Display summary
print("\nFeatures Summary (for features passing assumptions or transformed):")
print(features_summary)



Features Summary (for features passing assumptions or transformed):
      EEG_Exponent-parietal-r  EEG_Exponent-parietal-l  EEG_Offset-temporal-r  \
min                      0.53                     0.27                  -0.20   
max                      1.98                     1.91                   2.13   
mean                     1.33                     1.32                   0.90   

      EEG_Offset-temporal-l  EEG_Hurst-Frontal  EEG_Hurst-Central  \
min                    0.07               0.80               0.82   
max                    2.10               0.98               0.97   
mean                   0.92               0.91               0.90   

      EEG_Hurst-temporal-r  EEG_Hurst-temporal-l  EEG_Hurst-parietal-r  \
min                   0.78                  0.75                  0.80   
max                   0.97                  0.97                  0.97   
mean                  0.90                  0.90                  0.90   

      EEG_Hurst-parietal-l  ... 

Z scores

In [17]:
# Calculate z-scores for each feature within each diagnostic group
z_scores = pd.DataFrame()

for group in diagnostic_groups_data['diagnostic_group'].unique():
    group_data = diagnostic_groups_data[diagnostic_groups_data['diagnostic_group'] == group]
    
    # Calculate z-scores for features passing assumptions or transformed
    group_z_scores = group_data[features_to_use].apply(lambda x: (x - x.mean()) / x.std())
    
    # Add group identifier
    group_z_scores['diagnostic_group'] = group
    
    # Append to main z-scores dataframe
    z_scores = pd.concat([z_scores, group_z_scores])

# Reset index of final dataframe
z_scores = z_scores.reset_index(drop=True)

print("\nZ-scores calculated for each diagnostic group")
print(f"Shape of z-scores dataframe: {z_scores.shape}")




Z-scores calculated for each diagnostic group
Shape of z-scores dataframe: (80, 1563)


In [18]:
# Check for extreme z-scores (|z| > 3.29)
extreme_mask = (z_scores[features_to_use].abs() > 3.29)
num_extreme = extreme_mask.sum()

print("\nNumber of extreme z-scores (|z| > 3.29) for each EEG feature:")
print(num_extreme)

# Get columns with extreme scores
columns_with_extremes = [col for col, count in num_extreme.items() if count > 0]
print("\nColumns with extreme scores:")
print(columns_with_extremes)

total_extreme_rows = extreme_mask.any(axis=1).sum()

if total_extreme_rows > 0:
    print(f"\nFound {total_extreme_rows} rows with extreme z-scores")
    print("\nBreakdown by diagnostic group:")
    for group in [0, 1, 2]:
        group_rows = extreme_mask[z_scores['diagnostic_group'] == group].any(axis=1).sum()
        group_name = {
            0: "Control",
            1: "Neurodev only", 
            2: "Genetic carrier"
        }[group]
        print(f"{group_name}: {group_rows} rows with extreme values")
else:
    print("\nNo extreme z-scores found.")



Number of extreme z-scores (|z| > 3.29) for each EEG feature:
EEG_Exponent-parietal-r     0
EEG_Exponent-parietal-r     0
EEG_Exponent-parietal-r     0
EEG_Exponent-parietal-r     0
EEG_Exponent-parietal-r     0
                           ..
EEG_DFA-Alpha_temporal-l    0
EEG_DFA-Alpha_Occipital     0
EEG_DFA-Beta_parietal-r     0
EEG_DFA-Beta_parietal-l     0
EEG_DFA-Beta_Occipital      0
Length: 16852, dtype: int64

Columns with extreme scores:
['EEG_Delta-temporal-r', 'EEG_Delta-temporal-r', 'EEG_Delta-temporal-r', 'EEG_Delta-temporal-r', 'EEG_Delta-temporal-r', 'EEG_Delta-temporal-r', 'EEG_Delta-temporal-r', 'EEG_Delta-temporal-r', 'EEG_Delta-temporal-r', 'EEG_Delta-temporal-r', 'EEG_Delta-temporal-r', 'EEG_Delta-temporal-l', 'EEG_Delta-temporal-l', 'EEG_Delta-temporal-l', 'EEG_Delta-temporal-l', 'EEG_Delta-temporal-l', 'EEG_Delta-temporal-l', 'EEG_Delta-temporal-l', 'EEG_Delta-temporal-l', 'EEG_Delta-temporal-l', 'EEG_Delta-temporal-l', 'EEG_Delta-temporal-l', 'EEG_Theta-temporal-

Adjust extreme scores (Z > 3.29)

In [19]:
def adjust_extreme_scores(db, colname):
    # Convert list to array if needed
    if isinstance(db[colname], list):
        db[colname] = np.array(db[colname])
    
    # Convert to numeric
    db[colname] = pd.to_numeric(db[colname])
    
    # Calculate min/max thresholds
    mean = np.nanmean(db[colname])
    std = np.nanstd(db[colname])
    min_val = mean - (3.29 * std)
    max_val = mean + (3.29 * std)
    
    # Replace extreme scores
    db[colname] = np.where(
        pd.isna(db[colname]), 
        np.nan,
        np.where(
            db[colname] < min_val,
            min_val,
            np.where(
                db[colname] > max_val,
                max_val,
                db[colname]
            )
        )
    )
    
    return db

In [20]:
df = diagnostic_groups_data.copy()

# Adjust extreme scores for each column with extreme z-scores
for col in columns_with_extremes:
    df = adjust_extreme_scores(df, col)

Descriptive stats

In [21]:
# Get numeric columns but exclude binary diagnostic columns
numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col]) 
                and not col.startswith('diag_')
                and not col.startswith('inheritance_') 
                and col != ('diagnostic_group')
                and col != ('EEG_attempted')
                and col != ('EEG_site')
                and col != ('EEG_date')]
descriptive_stats = df[numeric_cols].describe()

# Add skewness and kurtosis
descriptive_stats.loc['skew'] = df[numeric_cols].skew()
descriptive_stats.loc['kurtosis'] = df[numeric_cols].kurtosis()

# Display the descriptive statistics
print("\nDescriptive Statistics:")
print(descriptive_stats)

# Get descriptive statistics for EEG_age by diagnostic group
age_stats = df.groupby('diagnostic_group')['EEG_age'].describe()
age_stats.index = ['Control', 'Neurodevelopmental', 'Genetic Carrier']

# Get sex counts by diagnostic group
sex_counts = pd.crosstab(df['diagnostic_group'], df['Sex_at_birth'])
sex_counts.index = ['Control', 'Neurodevelopmental', 'Genetic Carrier']

print("\nEEG Age Statistics by Group:")
print("-" * 50)
print(age_stats)

print("\nSex Distribution by Group:")
print("-" * 50)
print(sex_counts)




Descriptive Statistics:
            EEG_age  Affected_chromosome  Proximal_boundary  Distal_boundary  \
count     80.000000            19.000000       2.000000e+01     2.000000e+01   
mean      26.312802             9.473684       7.796647e+07     8.632170e+07   
std       16.356527             7.647573       5.722259e+07     5.770933e+07   
min        2.981581             1.000000       6.962810e+05     1.442861e+06   
25%       10.256171             2.000000       3.187610e+07     4.554903e+07   
50%       31.677584             7.000000       5.952243e+07     6.991131e+07   
75%       40.278719            15.500000       1.459756e+08     1.485300e+08   
max       58.133865            22.000000       1.921322e+08     1.952685e+08   
skew       0.102806             0.471302       5.461016e-01     3.365430e-01   
kurtosis  -1.474440            -1.155652      -1.015935e+00    -1.264128e+00   

          NVIQ_CIupr     ORASD_upr   SRS_CIupr  PdN_CIupr  sum_LOEUF_complete  \
count      14

In [22]:
# Create a combined IQ score column by merging WAIS and WISC scores
df['IQ_score'] = df['wais_globalapt_comp'].fillna(df['wisc_gai_is'])

# Get descriptive statistics for combined IQ scores by diagnostic group
iq_stats = df.groupby('diagnostic_group')['IQ_score'].describe()
iq_stats.index = ['Control', 'Neurodevelopmental', 'Genetic Carrier']

print("\nCombined IQ Score Statistics by Group:")
print("-" * 50)
print(iq_stats)



Combined IQ Score Statistics by Group:
--------------------------------------------------
                    count        mean        std   min    25%    50%     75%  \
Control              20.0  105.200000  14.993332  79.0  98.50  103.0  117.50   
Neurodevelopmental   24.0   98.916667  18.945957  65.0  85.25  101.0  107.00   
Genetic Carrier      16.0   94.937500  17.908913  58.0  85.00   95.0  102.75   

                      max  
Control             130.0  
Neurodevelopmental  134.0  
Genetic Carrier     128.0  


# Main Analysis

## ANCOVA

In [23]:
anova_results = {}

eeg_cols = df[features_to_use]

# Define diagnostic groups
diagnostic_groups = [0, 1, 2]  # 0=control, 1=neurodev, 2=genetic carrier
group_labels = ['Control', 'Neurodevelopmental', 'Genetic Carrier']

# Convert Sex_at_birth to numeric (0=Male, 1=Female)
df['Sex_at_birth'] = (df['Sex_at_birth'] == 'Female').astype(int)

# Convert EEG columns to numeric type and handle any string values
for col in eeg_cols:
    df[col] = pd.to_numeric(df[col].replace(['', 'NA', 'nan'], np.nan), errors='coerce').astype('float64')
    valid_count = df[col].notna().sum()
    print(f"{col}: {valid_count} valid numeric values")

print("\nANCOVA Results:")
print("-" * 50)

for eeg_feature in eeg_cols:
    print(f"\nAnalyzing {eeg_feature}")

    # Prepare data for analysis
    analysis_df = df[[eeg_feature, 'diagnostic_group', 'Sex_at_birth', 'EEG_age', 'IQ_score']].dropna()
    analysis_df[eeg_feature] = analysis_df[eeg_feature].astype('float64')

    # Print group sizes
    for group, label in zip(diagnostic_groups, group_labels):
        group_size = len(analysis_df[analysis_df['diagnostic_group'] == group])
        print(f"{label} group size: {group_size}")

    # Skip if any group has no data
    if any(len(analysis_df[analysis_df['diagnostic_group'] == group]) == 0 for group in diagnostic_groups):
        print(f"Skipping {eeg_feature} - insufficient data in one or more groups")
        continue

    try:
        # Fit ANCOVA using formula API (handles categorical encoding + covariates)
        # formula = f'Q("{eeg_feature}") ~ C(diagnostic_group) + Sex_at_birth + EEG_age' # Comment out to compare with IQ as covariable
        formula = f'Q("{eeg_feature}") ~ C(diagnostic_group) + Sex_at_birth + EEG_age + IQ_score' 
        model = ols(formula, data=analysis_df).fit()

        # Compute ANOVA table using Type II SS
        anova_table = sm.stats.anova_lm(model, typ=2)

        # Extract and store diagnostic group effect
        f_val = anova_table.loc['C(diagnostic_group)', 'F']
        p_val = anova_table.loc['C(diagnostic_group)', 'PR(>F)']
        anova_results[eeg_feature] = {
            'F-statistic': f_val,
            'p-value': p_val
        }

        print(f"\nFeature: {eeg_feature}")
        print(f"F-statistic: {f_val:.4f}")
        print(f"p-value: {p_val:.4f}")
        print("\nANOVA Table:")
        print(anova_table)

    except Exception as e:
        print(f"\nError analyzing {eeg_feature}: {str(e)}")

# Save ANOVA summary to file
with open(os.path.join(root_dir, 'Output/SPR-2025/anova_results_with_IQ.txt'), 'w') as f:
    f.write("ANOVA Results Summary\n")
    f.write("====================\n\n")

    for eeg_feature in anova_results:
        f.write(f"\nFeature: {eeg_feature}\n")
        f.write(f"F-statistic: {anova_results[eeg_feature]['F-statistic']:.4f}\n")
        f.write(f"p-value: {anova_results[eeg_feature]['p-value']:.4f}\n")
        f.write("\n" + "-" * 50 + "\n")

print("\n✅ ANCOVA results exported to 'anova_results_with_IQ.txt'")


EEG_Exponent-parietal-r: 80 valid numeric values
EEG_Exponent-parietal-l: 80 valid numeric values
EEG_Offset-temporal-r: 80 valid numeric values
EEG_Offset-temporal-l: 80 valid numeric values
EEG_Hurst-Frontal: 80 valid numeric values
EEG_Hurst-Central: 80 valid numeric values
EEG_Hurst-temporal-r: 80 valid numeric values
EEG_Hurst-temporal-l: 80 valid numeric values
EEG_Hurst-parietal-r: 80 valid numeric values
EEG_Hurst-parietal-l: 80 valid numeric values
EEG_Hurst-Occipital: 80 valid numeric values
EEG_Hurst-WholeBrain: 80 valid numeric values
EEG_Delta-Frontal_log: 80 valid numeric values
EEG_Delta-Central_log: 80 valid numeric values
EEG_Delta-temporal-r: 80 valid numeric values
EEG_Delta-temporal-l: 80 valid numeric values
EEG_Delta-parietal-r_log: 80 valid numeric values
EEG_Delta-parietal-l_log: 80 valid numeric values
EEG_Delta-Occipital_log: 80 valid numeric values
EEG_Delta-WholeBrain_log: 80 valid numeric values
EEG_Theta-Frontal_log: 80 valid numeric values
EEG_Theta-Centr

Post-Hoc

In [24]:
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import MultiComparison
import statsmodels.api as sm
from itertools import combinations

anova_df = pd.DataFrame(anova_results).T  # Transpose to make features as rows
anova_df.sort_values('p-value', inplace=True)

# --- Filter significant features ---
signif_features = anova_df[anova_df['p-value'] < 0.05].index.tolist()

# Output file
output_file = os.path.join(root_dir, 'Output/SPR-2025/tukey_adjusted_results_with_IQ.txt')
with open(output_file, 'w') as f:
    f.write("Estimated Marginal Means Post-Hoc Tests (adjusted for EEG_age, Sex_at_birth, and IQ_score)\n")
    f.write("=" * 70 + "\n\n")

    for feature in signif_features:
        print(f"\n📊 Adjusted post-hoc comparisons for: {feature}")

        # Drop rows with NaN values for this analysis
        analysis_df = df.dropna(subset=[feature, 'EEG_age', 'Sex_at_birth', 'IQ_score'])

        # Fit ANCOVA model using formula
        formula = f'Q("{feature}") ~ C(diagnostic_group) + EEG_age + Sex_at_birth + IQ_score'
        model = ols(formula, data=analysis_df).fit()

        # Compute group-wise adjusted means (EMMs)
        means = []
        for group in [0, 1, 2]:
            pred_df = analysis_df.assign(
                diagnostic_group=group,
                EEG_age=analysis_df['EEG_age'].mean(),
                Sex_at_birth=analysis_df['Sex_at_birth'].mean(),
                IQ_score=analysis_df['IQ_score'].mean()
            )
            group_mean = model.predict(pred_df).mean()
            means.append(group_mean)

        group_means = dict(zip(['Control', 'Neurodev', 'Genetic'], means))

        f.write(f"Feature: {feature}\n")
        f.write("-" * 50 + "\n")

        # Do pairwise t-tests between adjusted group predictions
        for (g1, g2) in combinations([0, 1, 2], 2):
            name1 = ['Control', 'Neurodev', 'Genetic'][g1]
            name2 = ['Control', 'Neurodev', 'Genetic'][g2]

            # Subset data
            df1 = analysis_df[analysis_df['diagnostic_group'] == g1].copy()
            df2 = analysis_df[analysis_df['diagnostic_group'] == g2].copy()

            if len(df1) > 0 and len(df2) > 0:
                # Predict adjusted values for each group using model
                df1_pred = model.predict(df1.assign(diagnostic_group=g1))
                df2_pred = model.predict(df2.assign(diagnostic_group=g2))

                # Run t-test on predicted values (adjusted outcomes)
                t_stat, p_val = stats.ttest_ind(df1_pred, df2_pred)

                result_str = f"{name1} vs {name2}:\n  t-stat: {t_stat:.4f}, p-value: {p_val:.4f}\n"
            else:
                result_str = f"{name1} vs {name2}:\n  Insufficient data after removing NaN values\n"
            
            print(result_str)
            f.write(result_str)

        f.write("=" * 70 + "\n")

print(f"\n📁 Adjusted Tukey-style results exported to:\n{output_file}")



📊 Adjusted post-hoc comparisons for: EEG_Alpha-Frontal_log
Control vs Neurodev:
  t-stat: -3.4014, p-value: 0.0015

Control vs Genetic:
  t-stat: -8.5281, p-value: 0.0000

Neurodev vs Genetic:
  t-stat: -4.3894, p-value: 0.0001


📊 Adjusted post-hoc comparisons for: EEG_PeriodicAlpha-Occipital_log
Control vs Neurodev:
  t-stat: 0.4299, p-value: 0.6694

Control vs Genetic:
  t-stat: -9.7440, p-value: 0.0000

Neurodev vs Genetic:
  t-stat: -8.5715, p-value: 0.0000


📊 Adjusted post-hoc comparisons for: EEG_relative-HighGamma-Occipital_log
Control vs Neurodev:
  t-stat: -0.0008, p-value: 0.9993

Control vs Genetic:
  t-stat: 6.7707, p-value: 0.0000

Neurodev vs Genetic:
  t-stat: 5.6584, p-value: 0.0000


📊 Adjusted post-hoc comparisons for: EEG_DFA-Beta_Occipital
Control vs Neurodev:
  t-stat: 3.1062, p-value: 0.0034

Control vs Genetic:
  t-stat: -7.0380, p-value: 0.0000

Neurodev vs Genetic:
  t-stat: -9.1668, p-value: 0.0000


📊 Adjusted post-hoc comparisons for: EEG_PeriodicTheta-Oc

#### Visualize

In [25]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from statsmodels.formula.api import ols
from scipy import stats
from itertools import combinations

def visualize_significant_results(df, anova_results, group_labels, feature_groups, output_dir=None):
    os.makedirs(output_dir, exist_ok=True)

    # Filter significant features
    signif_df = pd.DataFrame(anova_results).T
    signif_df = signif_df[signif_df['p-value'] < 0.05].sort_values('p-value')
    signif_features = signif_df.index.tolist()

    print(f"\n🎯 Visualizing {len(signif_features)} significant features...")

    # --------- GROUP SIGNIFICANT FEATURES BY TYPE ---------
    feature_type_map = {}
    for group_name, feature_list in feature_groups.items():
        for f in feature_list:
            # Handle both regular and log-transformed features
            feature_type_map[f] = group_name
            feature_type_map[f + '_log'] = group_name

    grouped_signif_features = {}
    for feature in signif_features:
        # Strip _log suffix when looking up feature type
        base_feature = feature.replace('_log', '')
        group = feature_type_map.get(base_feature, "Other")
        grouped_signif_features.setdefault(group, []).append(feature)

    # --------- PLOT BOXPLOTS PER FEATURE TYPE ---------
    for group_name, group_features in grouped_signif_features.items():
        n_cols = 2
        n_rows = int(np.ceil(len(group_features) / n_cols))
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 5 * n_rows))
        axes = axes.flatten()

        print(f"📂 Plotting {len(group_features)} features for: {group_name}")

        for i, feature in enumerate(group_features):
            ax = axes[i]
            analysis_df = df[[feature, 'diagnostic_group', 'EEG_age', 'Sex_at_birth', 'IQ_score']].dropna()

            # Fit ANCOVA model
            formula = f'Q("{feature}") ~ C(diagnostic_group) + EEG_age + Sex_at_birth + IQ_score'
            model = ols(formula, data=analysis_df).fit()

            # Use raw data
            adjusted_df = analysis_df.copy()
            adjusted_df['adjusted'] = adjusted_df[feature]

            # Plot boxplot and points
            sns.boxplot(data=adjusted_df, x='diagnostic_group', y='adjusted', ax=ax)
            sns.stripplot(data=adjusted_df, x='diagnostic_group', y='adjusted', color='black', alpha=0.3, ax=ax)
            ax.set_title(feature, fontsize=10)
            ax.set_xticks([0, 1, 2])
            ax.set_xticklabels(group_labels)
            ax.set_xlabel("")
            ax.set_ylabel("Raw Value")

            # Pairwise post-hoc comparisons
            comparisons = list(combinations([0, 1, 2], 2))
            y_max = adjusted_df['adjusted'].max()
            height_step = (y_max - adjusted_df['adjusted'].min()) * 0.1
            current_height = y_max + height_step

            for g1, g2 in comparisons:
                df1 = adjusted_df[adjusted_df['diagnostic_group'] == g1]['adjusted']
                df2 = adjusted_df[adjusted_df['diagnostic_group'] == g2]['adjusted']
                t_stat, p_val = stats.ttest_ind(df1, df2)

                if p_val < 0.01:  # Significant at p < 0.01
                    ax.plot([g1, g1, g2, g2], [current_height, current_height + 0.05,
                                               current_height + 0.05, current_height],
                            lw=1.2, color='black')
                    ax.text((g1 + g2) / 2, current_height + 0.06, "**", ha='center', va='bottom', color='black')
                    current_height += height_step
                elif p_val < 0.05:  # Significant at p < 0.05
                    ax.plot([g1, g1, g2, g2], [current_height, current_height + 0.05,
                                               current_height + 0.05, current_height],
                            lw=1.2, color='black')
                    ax.text((g1 + g2) / 2, current_height + 0.06, "*", ha='center', va='bottom', color='black')
                    current_height += height_step

        # Remove any unused subplots
        for j in range(i + 1, len(axes)):
            fig.delaxes(axes[j])

        plt.tight_layout()
        group_filename = f"{group_name.replace(' ', '_')}_features_boxplots_with_posthoc_with_IQ.png"
        plt.savefig(os.path.join(output_dir, group_filename))
        plt.close()

    print("✅ Boxplots by feature group saved.")

    # --------- HEATMAP OF ADJUSTED GROUP MEANS ---------
    adjusted_means_matrix = pd.DataFrame(index=signif_features, columns=group_labels)

    for feature in signif_features:
        analysis_df = df[[feature, 'diagnostic_group', 'EEG_age', 'Sex_at_birth', 'IQ_score']].dropna()
        model = ols(f'Q("{feature}") ~ C(diagnostic_group) + EEG_age + Sex_at_birth + IQ_score', data=analysis_df).fit()

        for group_code, group_name in enumerate(group_labels):
            group_df = analysis_df.copy()
            group_df['diagnostic_group'] = group_code
            group_df['EEG_age'] = group_df['EEG_age'].mean()
            group_df['Sex_at_birth'] = group_df['Sex_at_birth'].mean()
            group_df['IQ_score'] = group_df['IQ_score'].mean()
            group_df['adjusted'] = model.predict(group_df)
            adjusted_means_matrix.loc[feature, group_name] = group_df['adjusted'].mean()

    adjusted_means_matrix = adjusted_means_matrix.astype(float)

    plt.figure(figsize=(8, len(signif_features) * 0.3 + 2))
    sns.heatmap(adjusted_means_matrix, annot=True, cmap="vlag", cbar=True, linewidths=0.5, fmt=".2f")
    plt.title("Estimated Marginal Means by Group")
    plt.ylabel("EEG Feature")
    plt.xlabel("Group")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "adjusted_means_heatmap_with_IQ.png"))
    plt.close()

    print("✅ Adjusted means heatmap saved.")


In [26]:
visualize_significant_results(df, anova_results, group_labels, feature_groups, output_dir=os.path.join(root_dir, 'Output/SPR-2025/plots/IQ'))


🎯 Visualizing 42 significant features...
📂 Plotting 8 features for: Alpha
📂 Plotting 9 features for: Periodic
📂 Plotting 15 features for: Relative
📂 Plotting 3 features for: Beta
📂 Plotting 4 features for: Exponent
📂 Plotting 1 features for: Theta
📂 Plotting 1 features for: Low Gamma
📂 Plotting 1 features for: High Gamma
✅ Boxplots by feature group saved.
✅ Adjusted means heatmap saved.
